In [1]:
from openai import OpenAI
import os
import json
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
def get_chatbot_response(client,model_name,messages,temperature=0):
    input_messages = []
    for message in messages:
        input_messages.append({"role":message["role"],"content":message["content"]})
    response = client.chat.completions.create(
        model=model_name,
        messages=input_messages,
        temperature=temperature, #amount of randomness
        top_p=0.8,
        max_tokens=2000
    ).choices[0].message.content
    return response

In [2]:
#connecting to llama endpoint
client = OpenAI(
    api_key=os.getenv('RUNPOD_TOKEN'),
    base_url=os.getenv('RUNPOD_CHATBOT_URL'),
)
model_name=os.getenv("MODEL_NAME")

# Get LLM Response

In [ ]:
messages = [{"role":"system","content":"What's the capital of Germany?"}]
response= get_chatbot_response(client,model_name,messages)

In [ ]:
response

# Prompt Engineering 
#### Guiding the chatbot with precise instructions.

## Structured Output Technique

In [ ]:
system_prompt="""
You are a helpful assisstant that answer questions about capitals of countries

Your output should be in a structured json format exactly like the one bellow. You are not allowed to write anything other than the json object:
[
{
    "country": the country that you will get the capital of
    "capital": the capital of the country stated
}
]
"""

messages = [{"role":"system","content":system_prompt}]
messages.append({"role":"user","content":"What's the capital of Germany?"})
message = get_chatbot_response(client,model_name,messages)
print(response)

In [ ]:
type(response)

In [ ]:
json_response = json.loads(response)
json_response

In [ ]:
type(json_response[0]),json_response[0]['capital']

## Input Structuring Technique

In [ ]:
user_input = """
Get me the capitals of the following countries:
```
1. Italy
2. Germany
3. France
```
"""

messages = [{"role":"system","content":system_prompt}]
messages.append({"role":"user","content":user_input})
response = get_chatbot_response(client,model_name,messages)
print(response)

In [ ]:
json_response = json.loads(response)
json_response

## Give the model time to think (Chain of thought or CoT technique)

In [ ]:
user_prompt = """
Calculate the result of this equation: 259/2*8654+91072*33-12971

Your output should be in a structured json format exactly like the one below. You are not allowed to write anything other than the json object:
{
    steps: This is where you solve the equation bit by bit following the BEDMAS order of operations. You need to show your work and calculate each step leading to the final result. Feel free to write in free text.
    result: The final number resulted from calculating the equation above
}
"""

messages = [{"role":"user","content":user_prompt}]
response = get_chatbot_response(client,model_name,messages)
print(response)

# Retrieval-Augmented Generation (RAG)
#### Enhance chatbot answers using personalized data.

In [ ]:

iphone_16 = """
The iPhone 16 introduces several exciting updates, making it one of Apple's most advanced smartphones to date. It features a larger 6.1-inch display for the base model and a 6.7-inch screen for the iPhone 16 Plus, with thinner bezels and a more durable Ceramic Shield. The iPhone 16 Pro and Pro Max boast even larger displays, measuring 6.3 and 6.9 inches respectively, offering the thinnest bezels seen on any Apple product so far.

Powered by the new A18 chip (A18 Pro for the Pro models), these phones deliver significant performance improvements, with enhanced neural engine capabilities, faster GPU for gaming, and machine learning tasks. The camera systems are also upgraded, with the base iPhone 16 sporting a dual-camera setup with a 48MP main sensor. The Pro models offer a 48MP Ultra Wide and 5x telephoto camera, enhanced by Apple’s "Camera Control" button for more flexible photography options.

Apple also introduced advanced audio features like "Audio Mix," which uses machine learning to separate background sounds from speech, allowing for more refined audio capture during video recording. Battery life has been extended, especially in the iPhone 16 Pro Max, which is claimed to have the longest-lasting battery of any iPhone 
9TO5MAC

APPLEMAGAZINE
.

Additionally, Apple has switched to USB-C for faster charging and data transfer, and the Pro models now support up to 2x faster video encoding. The starting prices remain consistent with previous generations, with the iPhone 16 starting at $799, while the Pro models start at $999
"""


In [ ]:
user_prompt = f"""
{iphone_16}

What's new in iphone 16?
"""

messages = [{'role':'user','content':user_prompt}]
response = get_chatbot_response(client,model_name,messages)
print(response)

In [ ]:
#Embeddings